In [1]:
import os

import torch

['mae-multiscale', 'rt-detr']

In [44]:
folder = "/home/simon/Desktop/yolo-training/SportsInnovation Game1/job_2915-2026_03_10_22_43_36-yolo 1.1/obj_train_data"

all_files = os.listdir(folder)
imgs = [f for f in all_files if f.endswith('.png')]
txts = [f for f in all_files if f.endswith('.txt')]

names_file = "/home/simon/Desktop/yolo-training/CVAT Log/obj.names"

with open(names_file, 'r') as f:
    name_conversion = {name.removesuffix("\n"): i for i, name in enumerate(f.readlines())}
    new_ids = {"ball": 0,
               "robot": 1,
               "penalty cross": 2}

# First value of tuple is replaced by second value
replaces = [("SPL Ball", "ball"),
            ("FIFA 26 Ball", "ball"),
            # ("Nao", "robot"),
            ("K1", "robot"),
            ("PenaltyMark", "penalty cross")]
replaces_int = [(name_conversion[old], new_ids[new]) for old, new in replaces]



In [45]:
count = 0
for t in txts:
    with open(os.path.join(folder, t), 'r') as f:
        lines = f.readlines()

    new_lines = []
    for line in lines:
        parts = line.split()
        class_id = int(parts[0])
        if class_id in [0, 1, 3, 5]:
            for old_id, new_id in replaces_int:
                if class_id == old_id:
                    parts[0] = str(new_id)
                    # print(f"Replaced class ID {old_id} with {new_id} in file {t}")
                    break
            new_lines.append(" ".join(parts))

    with open(os.path.join(folder, t), 'w') as f:
        f.write("\n".join(new_lines))
        count+=1
print(f"Processed {count} label files.")

Processed 50 label files.


# Fill missing empty labels

In [10]:
img_folder = "/home/simon/Desktop/yolo-training/SportsInnovation Game1/job_2917-2026_03_11_12_12_46-yolo 1.1/obj_train_data"
lbl_folder = ""
imgs = [f for f in os.listdir(img_folder) if f.endswith('.png')]
txts = [f for f in os.listdir(lbl_folder) if f.endswith('.txt')]

count = 0
for img in imgs:
    txt_file = img.replace('.png', '.txt')
    if txt_file not in txts:
        with open(os.path.join(folder, txt_file), 'w') as f:
            f.write("")
            count+=1
print(f"Created {count} empty label files.")

Created 10 empty label files.


# Replace IDs (back from 3cls to 11cls)

In [27]:
# path = "/home/simon/Downloads/job_2915-2026_03_10_11_40_52-yolo 1.1/obj_train_data"

# Fist gets replaced by second value
replaces = {0: 0,
            1: 3,
            2: 5}

label_files = [f for f in all_files if f.endswith('.txt')] #[p for p in os.listdir(path) if p.endswith(".txt")]
for label_file in label_files:
    with open(os.path.join(folder, label_file), 'r') as f:
        lines = f.readlines()

    new_lines = []
    for line in lines:
        parts = line.split()
        parts[0] = str(replaces[int(parts[0])])
        new_lines.append(" ".join(parts))

    with open(os.path.join(folder, label_file), 'w') as f:
        f.write("\n".join(new_lines))

# Convert Robert format to YOLO format

In [60]:
from pathlib import Path

path = "/home/simon/Desktop/yolo-training/RobertData/labels"
out_path = "/home/simon/Desktop/yolo-training/RobertData/labels_yolo"
Path(out_path).mkdir(parents=True, exist_ok=True)
label_files = [f for f in os.listdir(path) if f.endswith('.txt')]

In [63]:
def parse_bbox_line(line):
    line_split = line.split(":")
    cls, coords = line_split[0].strip(), line_split[1].strip()
    class_id = name_to_id[cls]

    if len(coords.split(" ")) != 4:
        print(f"Skipping line in file {label_file} due to incorrect format: {line}")
        return ""
    new_line = f"{class_id} {coords}"
    return new_line

def parse_ppoint_line(line):
    if len(line.split(" ")) != 4:
        print(f"Skipping line in file {label_file} due to incorrect format: {line}")
        return ""
    new_line = f"2 {line.strip()}"
    return new_line

In [64]:
name_to_id = {
    "Trionda Ball 2026(Clone)": 0,
    "K1(Clone)": 1
}

headers = ["BoundingBoxes", "GoalPosts", "CenterCircle", "PenaltyPoints", "Lines"]  # Add more headers if needed
currentSection = ""

for label_file in label_files:
    with open(os.path.join(path, label_file), 'r') as f:
        lines = f.readlines()

    new_lines = []
    for line in lines:
        if line.split(":")[0] in headers:
            currentSection = line.split(":")[0]
            continue

        match currentSection:
            case "BoundingBoxes":
                nl = parse_bbox_line(line)
                if nl != "":
                    new_lines.append(nl)
            case "GoalPosts":
                pass
            case "CenterCircle":
                pass
            case "PenaltyPoints":
                nl = parse_ppoint_line(line)
                if nl != "":
                    new_lines.append(nl)
            case "Lines":
                pass
            case _:
                print(f"Unknown section {currentSection} in file {label_file}")
                continue

    with open(os.path.join(out_path, label_file), 'w') as f:
        f.write("\n".join(new_lines))

# Convert datumaro to YOLO oBB

In [10]:
import json
dataset = "/home/simon/Desktop/yolo-training/task_booster k1 log 02-26-2026_03_08_09_50_02-datumaro 1.0"
datumaro_file = os.path.join(dataset, "annotations/default.json")
img_path = "images/default"
norm_vec_len = 5

with open(datumaro_file, 'r') as f:
    data = json.load(f)


label_to_id = {name: idx for idx, name in enumerate([name["name"] for name in data["categories"]["label"]["labels"]])}

In [9]:
def fix_daniel_fleck(data):
    for img_ind, img_data in enumerate(data['items']):
        anns_to_be_added = []
        for ann_ind, ann in enumerate(img_data['annotations']):
            if ann['type'] == "polyline":
                points = ann['points']
                if len(points) != 4:
                    print(f"Fixing Polyline for img: {img_data['id']} with {len(points)/2-1} lines")
                    for i in range(0, len(points)-3, 2):
                        p1 = (points[i], points[i+1])
                        p2 = (points[i+2], points[i+3])
                        new_ann = ann.copy()
                        new_ann["points"] = [p1[0], p1[1], p2[0], p2[1]]
                        if not (p1[0] == p2[0] and p1[1] == p2[1]):  # Check for zero-length vector
                            anns_to_be_added.append(new_ann)
                        else:
                            print(f"Skipping double point for annotation {ann_ind} in file {img_data['id']} ({img_ind}) due to zero-length vector: {p1} and {p2}")

                    ann['points'] = anns_to_be_added[0]['points']
                    anns_to_be_added = anns_to_be_added[1:]
        img_data['annotations'].extend(anns_to_be_added)

In [27]:
# fix_daniel_fleck(data)
# datumaro_to_devilsYoloOBB(data)
datumaro_to_devilsYoloAABB(data)

In [6]:
def datumaro_to_devilsYoloOBB(data, normalize=(544, 448)):
    for img_ind, img_data in enumerate(data['items']):
        with (open(os.path.join(dataset, img_path, img_data['id'] + ".txt"), 'w') as f):

            for ann_ind, ann in enumerate(img_data['annotations']):
                if ann['type'] == "bbox":
                    x1 = ann['bbox'][0] / normalize[0]
                    y1 = ann['bbox'][1] / normalize[1]
                    x2 = x1 + ann['bbox'][2] / normalize[0]
                    y2 = y1 / normalize[1]
                    x3 = x2 / normalize[0]
                    y3 = y1 + ann['bbox'][3] / normalize[1]
                    x4 = x1 / normalize[1]
                    y4 = y3 / normalize[0]
                    f.write(f"{ann['label_id']} {x1} {y1} {x2} {y2} {x3} {y3} {x4} {y4}\n")
                elif ann['type'] == "polyline":
                    points = ann['points']
                    if len(points) != 4:
                        print(f"Skipping annotation in file {img_data['id']} due to incorrect number of points: {points}")
                        continue
                    p1 = (points[0], points[1])
                    p2 = (points[2], points[3])

                    v = (p2[0] - p1[0], p2[1] - p1[1])
                    length = (v[0]**2 + v[1]**2)**0.5
                    if length == 0:
                        print(f"Skipping annotation {ann_ind} in file {img_data['id']}({img_ind}) due to zero-length vector: {v}")
                        continue
                    v_normal = (v[0]/(length/5) , v[1]/(length/5))

                    x1 = (p1[0] + v_normal[0]) / normalize[0]
                    y1 = (p1[1] + v_normal[1]) / normalize[1]
                    x2 = (p2[0] + v_normal[0]) / normalize[0]
                    y2 = (p2[1] + v_normal[1]) / normalize[1]
                    x3 = (p2[0] - v_normal[0]) / normalize[0]
                    y3 = (p2[1] - v_normal[1]) / normalize[1]
                    x4 = (p1[0] - v_normal[0]) / normalize[0]
                    y4 = (p1[1] - v_normal[1]) / normalize[1]

                    eps = 1e-2
                    x1 = max(min(1-eps, x1), 0+eps)
                    y1 = max(min(1-eps, y1), 0+eps)
                    x2 = max(min(1-eps, x2), 0+eps)
                    y2 = max(min(1-eps, y2), 0+eps)
                    x3 = max(min(1-eps, x3), 0+eps)
                    y3 = max(min(1-eps, y3), 0+eps)
                    x4 = max(min(1-eps, x4), 0+eps)
                    y4 = max(min(1-eps, y4), 0+eps)

                    if max(x1, y1, x2, y2, x3, y3, x4, y4) > 1 or min(x1, y1, x2, y2, x3, y3, x4, y4) < 0:
                        print(f"Skipping annotation {ann_ind} in file {img_data['id']}({img_ind}) due to out-of-bounds coordinates: {(x1, y1, x2, y2, x3, y3, x4, y4)}")
                        continue

                    f.write(f"{ann['label_id']} {x1} {y1} {x2} {y2} {x3} {y3} {x4} {y4}\n")

In [26]:
def dist(p1, p2):
    """
    Calculates Euclidean distance between two points.
    """
    return ((p1[0] - p2[0])**2 + (p1[1] - p2[1])**2)**0.5

def merge_points(points, max_eps):
    """
        Merges points that are closer to each other than max_eps
        :param points: List of points to be merged
        :param max_eps: Max number of points to merge

        :return: List of merged points
    """
    merged = []
    for p in points:
        found_merge = False
        for mp in merged:
            if dist(p, mp) < max_eps:
                mp[0] = (mp[0] + p[0]) / 2
                mp[1] = (mp[1] + p[1]) / 2
                found_merge = True
                break
        if not found_merge:
            merged.append(list(p))
    return merged

def datumaro_to_devilsYoloAABB(data, normalize=(544, 448), max_eps = 0.01):
    """
    Convert Image annotations exported from cvat in datumaro JSON format into YOLO format for axis-aligned bounding boxes (AABB).
    Line endpoints are merged into linecrossings and represented as BBoxes with w=h=0.01
    """
    for img_ind, img_data in enumerate(data['items']):
        with (open(os.path.join(dataset, img_path, img_data['id'] + ".txt"), 'w') as f):
            line_crossing_candidates = []
            for ann_ind, ann in enumerate(img_data['annotations']):
                if ann['type'] == "bbox":
                    cx = ann['bbox'][0] / normalize[0]
                    cy = ann['bbox'][1] / normalize[1]
                    w = ann['bbox'][2] / normalize[0]
                    h = ann['bbox'][3] / normalize[0]
                    cx += 0.5*w
                    cy += 0.5*h
                    f.write(f"{ann['label_id']} {cx} {cy} {w} {h}\n")
                elif ann['type'] == "polyline":
                    points = ann['points']
                    if len(points) != 4:
                        print(f"Skipping annotation in file {img_data['id']} due to incorrect number of points: {points}")
                        continue
                    p1 = (points[0]/ normalize[0], points[1]/ normalize[1])
                    p2 = (points[2]/ normalize[0], points[3]/ normalize[1])
                    if dist(p1, p2) > max_eps:
                        line_crossing_candidates.extend([p1, p2])
            line_crossings = merge_points(line_crossing_candidates, max_eps)
            for p in line_crossings:
                f.write(f"{8} {p[0]} {p[1]} {0.01} {0.01}\n")


# sanitize webots data (full-image bounding boxes)b

In [24]:
import os

threshold = 0.95

path = "/home/simon/Desktop/maesy-training/data/WebotsDataset/train/labels"
files = os.listdir(path)
count = 0
line_count = 0
for file in files:
    new_lines = []
    with open(os.path.join(path, file), 'r') as f:
        lines = f.readlines()
        for line in lines:
            if line.split(" ")[0] != 3:
                line_count += 1
                x, y, x2, y2, = line.split(" ")[1:]

                w = float(x2) - float(x)
                h = float(y2) -float(y)
                area = float(w)*float(h)
                if area < threshold:
                    new_lines.append(line)
                else:
                    # print(f, area)
                    count+=1
            else:
                new_lines.append(line)
    with open(os.path.join(path, file), 'w') as f:
        f.write("".join(new_lines))
print("Num files: ", len(files))
print("Avg lines per file: ", line_count/len(files))
print("Total lines: ", line_count)
print("Suspicious boxes: ", count)

Num files:  950
Avg lines per file:  15.162105263157894
Total lines:  14404
Suspicious boxes:  0


# Remove specific class ids from label files

In [2]:
import os
from pathlib import Path

id_blacklist = [3, 4]

path = Path("/home/simon/Desktop/yolo-training/Robert+Webots")
label_paths = [path/"train/labels", path/"val/labels"]
for lp in label_paths:
    for label_file in os.listdir(lp):
        with open(os.path.join(lp, label_file), 'r') as f:
            lines = f.readlines()
        new_lines = []
        for line in lines:
            if int(line.split()[0]) not in id_blacklist:
                new_lines.append(line)
        with open(os.path.join(lp, label_file), 'w') as f:
            f.write("\n".join(new_lines))

# Transform from xyxy to cxcywh

In [11]:
import os
from pathlib import Path

id_blacklist = [3, 4]
# path = Path("/home/simon/Desktop/yolo-training/Robert+Webots")
# path = Path("/home/simon/Desktop/yolo-training/YOLO_noGP")
path = Path("/home/simon/Desktop/yolo-training/WebotsDataset")
label_paths = [path/"train/labels", path/"val/labels"]
for lp in label_paths:
    for label_file in os.listdir(lp):
        with open(os.path.join(lp, label_file), 'r') as f:
            lines = f.readlines()
        new_lines = []
        for line in lines:
            if int(line.split()[0]) in id_blacklist:
                continue
            parts = line.split()
            w = float(parts[3]) - float(parts[1])
            h = float(parts[4]) - float(parts[2])
            cx = float(parts[1]) + w/2
            cy = float(parts[2]) + h/2
            new_lines.append(f"{parts[0]} {cx} {cy} {w} {h}")
        with open(os.path.join(lp, label_file), 'w') as f:
            f.write("\n".join(new_lines))

# Testing Area

In [19]:
from maesy.dataset import MaesyDataset
from typing import Tuple
from pathlib import Path
from PIL import Image
from maesy.training.utils import collate_detection_fn
import torch
from torch.utils.data import DataLoader

In [38]:
class Patch:
    """
        A small representation of a patch that automatically loads and crops itself
    """
    def __init__(self, position_in_image: Tuple[int, int, int, int], original_image_path):
        self.position_in_image = position_in_image
        self.original_image_path = original_image_path

    def save_patch_to(self, save_path: Path):
        """
        Save the patch to the given path.
        Utilizes lazy-loading of image data only when it is requested
        """
        with Image.open(self.original_image_path) as img:
            w, h = img.size
            x1, x2 = self.position_in_image[0]*w, self.position_in_image[2]*w
            y1, y2 = self.position_in_image[1]*h, self.position_in_image[3]*h
            img.crop((x1, y1, x2, y2)).save(save_path)

In [11]:
dataset_dir = "/home/simon/Desktop/maesy-training/data/Cvat"
dataset = MaesyDataset(dataset_dir, split="val", annotation_type="detection")
loader=DataLoader(dataset, 1, shuffle=False, collate_fn = collate_detection_fn)

class_id = 1


Loaded 79 val images
------------------------------


In [39]:
patch_list = []
for idx, (imgs, targets) in enumerate(loader):
    path = dataset.get_image_path(idx)
    for img, objects in zip(imgs, targets): # TO accomodate batch_sizes
        temp=[]
        # print(*zip(objects["labels"], objects["boxes"]))
        for label, box in zip(objects["labels"], objects["boxes"]):
            if label.item()==class_id:
                patch_list.append(Patch(box.tolist(), path))


In [27]:
t = torch.tensor([0.1, 0.2, 0.3, 0.4])
t.tolist()

[0.10000000149011612,
 0.20000000298023224,
 0.30000001192092896,
 0.4000000059604645]

In [40]:
p = patch_list[0]
save_path = "/home/simon/Desktop/maesy-training/data/Cvat"
p.save_patch_to(Path(save_path)/"patch_0.png")